In [ ]:
def return_correlation_graph_for_day(
    returns_df: pd.DataFrame,
    tickers: List[str],
    target_idx: int,
    window: int = 5,
    min_edge_weight: float = 0.001,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build an edge index and weight matrix based on the absolute Pearson correlation
    of stock returns over a rolling window ending at target_idx.
    """
    n = len(tickers)

    # 1. Slice the rolling window up to the target index
    start_idx = max(0, target_idx - window + 1)
    window_returns = returns_df.iloc[start_idx : target_idx + 1][tickers]

    # 2. Compute the Pearson correlation matrix
    if len(window_returns) > 2:
        # fillna(0.0) handles cases with zero variance (e.g., flat lined prices)
        corr_mat = window_returns.corr().fillna(0.0).values
    else:
        # Fallback if there aren't enough lookback steps yet
        corr_mat = np.eye(n, dtype=np.float32)

    # 3. Use absolute correlation to define edge strength (0 to 1)
    mat = np.abs(corr_mat)

    # 4. Filter out weak connections based on threshold
    mask = mat >= min_edge_weight
    if not np.any(mask):
        return np.zeros((2, 0), dtype=np.int64), np.zeros((0,), dtype=np.float32)

    rows, cols = np.nonzero(mask)
    edges = np.stack([rows, cols], axis=0).astype(np.int64)
    weights = mat[rows, cols].astype(np.float32)

    return edges, weights

In [ ]:
class DynamicGraphDatasetNews(torch.utils.data.Dataset):
    def __init__(
        self,
        price_df: pd.DataFrame,
        news_df: pd.DataFrame,
        sentiment_df: pd.DataFrame,
        seq_len: int = 5,
        co_window: int = 5,
        mode: str = "regression",
    ):
        assert mode in ("regression", "classification")

        self.seq_len = seq_len
        self.co_window = co_window
        self.mode = mode

        prices = price_df.values.astype(np.float32)
        dates = pd.to_datetime(price_df.index)
        self.date_index = dates
        self.tickers = list(price_df.columns)

        T, N = prices.shape

        sentiment_df = sentiment_df.reindex(dates).fillna(0.0)
        svals = sentiment_df.values.astype(np.float32)

        # --------------------------------------------------
        # PRE-COMPUTE DAILY LOG RETURNS
        # --------------------------------------------------
        self.returns_df = np.log(
            (price_df + 1e-8) / (price_df.shift(1) + 1e-8)
        ).fillna(0.0)

        returns = self.returns_df.values.astype(np.float32)

           # ---- PRICE NORMALIZATION  ----
        W = 252
        VOL_WINDOW = 20
        T, N = prices.shape

        log_returns = np.zeros((T, N), dtype=np.float32)
        log_returns[1:] = np.log(
            (prices[1:] + 1e-8) / (prices[:-1] + 1e-8)
        )

        # Initialize arrays to the full length T
        norm_prices = np.zeros((T, N), dtype=np.float32)
        vol_norm = np.zeros((T, N), dtype=np.float32)
        targets = np.zeros((T, N), dtype=np.float32)

        # ---- PROCESSING ALL TIMESTEPS ----
        for t in range(T):
            # 1. Determine the window bounds
            # If t < W, use everything from 0 to t (Expanding)
            # If t >= W, use t-W+1 to t (Rolling)
            start_idx = max(0, t - W + 1)
            window = prices[start_idx : t + 1]

            # 2. Calculate statistics
            mean_t = window.mean(axis=0)
            std_t = window.std(axis=0) + 1e-6

            # 3. Normalize Current Price
            norm_prices[t] = (prices[t] - mean_t) / std_t

            # 4. Normalize Target Price (Price at t+1)
            # We can only do this if t < T-1
            if t < T - 1:
                targets[t] = (prices[t+1] - mean_t) / std_t


          # ----- Volatility -----
            vol_start = max(0, t - VOL_WINDOW + 1)
            return_window = log_returns[vol_start:t+1]

            # Annualize
            current_vol = return_window.std(axis=0) * np.sqrt(252) # Shape (N,)

            vol_norm[t] = current_vol

        # --------------------------------------------------
        # FEATURE CONSTRUCTION
        # --------------------------------------------------
        feature_list = []

        for t in range(T - 1):

            feat_t = np.stack(
                [
                    norm_prices[t],   # Normalized price
                    svals[t],         # Sentiment
                    vol_norm[t],      # Rolling volatility
                ],
                axis=1,
            )

            feature_list.append(feat_t.astype(np.float32))

        self.features = np.stack(feature_list, axis=0)
        self.targets = targets[:T - 1]

        self.valid_end_idx = list(range(self.seq_len - 1, T - 1))

        # --------------------------------------------------
        # GRAPH CONSTRUCTION (RETURN CORRELATION)
        # --------------------------------------------------
        self.edge_index_list = []
        self.edge_weight_list = []

        for t in range(T - 1):

            ei, ew = return_correlation_graph_for_day(
                returns_df=self.returns_df,
                tickers=self.tickers,
                target_idx=t,
                window=self.co_window,
                min_edge_weight=0.001,
            )

            # ------------------------------------------
            # EDGE WEIGHT NORMALIZATION
            # ------------------------------------------
            if ew is not None and len(ew) > 0:

                ew = ew.astype(np.float32)

                # Correlation is already bounded in [0,1]
                # Only perform Min-Max normalization
                ew_min = ew.min()
                ew_max = ew.max()

                if ew_max > ew_min:
                    ew = (ew - ew_min) / (ew_max - ew_min)
                else:
                    ew = np.zeros_like(ew)

                # Avoid zero-weight edges
                ew = 0.1 + 0.9 * ew

            self.edge_index_list.append(ei)
            self.edge_weight_list.append(ew)

    def __len__(self):
        return len(self.valid_end_idx)

    def __getitem__(self, idx):

        end_t = self.valid_end_idx[idx]
        start_t = end_t - (self.seq_len - 1)

        seq_feats = self.features[start_t:end_t + 1]
        seq_edge_idx = self.edge_index_list[start_t:end_t + 1]
        seq_edge_w = self.edge_weight_list[start_t:end_t + 1]

        target = self.targets[end_t]

        if self.mode == "classification":
            y = (target > 0).astype(np.int64)
        else:
            y = target.astype(np.float32)

        return {
            "seq_feats": torch.from_numpy(seq_feats),
            "seq_edge_index": seq_edge_idx,
            "seq_edge_weight": seq_edge_w,
            "target": torch.from_numpy(y),
        }